# Task 4 - UAT Planning and Test Scenario Design
## AI-Augmented Business Analyst Capstone
**Company:** Meridian Financial | **Scenario:** FCA KYC/AML Onboarding Remediation
**Inputs:** functional_requirements.csv, requirements_systems_mapping.csv
---
## Overview
Diana Vlasova (Lead Engineer) needs a UAT test strategy and full test scenario outlines for every Sprint 1 requirement. Each requirement needs minimum 3 scenarios: happy path, alternative path, failure path.

| Sprint 1 Scope | Req | Type |
|---|---|---|
| Mandatory BO Declaration | FR-001 | FR |
| Automated PEP/Sanctions Screening | FR-002 | FR |
| Identity Verification Evidence Capture | FR-003 | FR |
| Automated Compliance Audit Log | FR-004 | FR |
| Client Risk Rating at Application | FR-005 | FR |
| Audit Log Retention 5 Years | NFR-001 | NFR |
| API Response Time <= 30s | NFR-002 | NFR |
| WCAG 2.1 AA Accessibility | NFR-005 | NFR |


## 1. Load Inputs

In [ ]:
import pandas as pd
import os
os.makedirs('../outputs', exist_ok=True)
df_fr = pd.read_csv('../outputs/functional_requirements.csv')
df_mapping = pd.read_csv('../outputs/requirements_systems_mapping.csv')
print('Sprint 1 requirements:', len(df_fr[df_fr['priority'].isin(['Must Have','Should Have'])]))
print(df_fr[['req_id','title','priority']].to_string(index=False))

## 2. UAT Test Strategy

### Scope
- **In:** All Sprint 1 FRs and NFRs; integration testing across Web Portal, Microservice, Salesforce FSC, Experian, Dow Jones
- **Out:** Sprint 2 requirements; regression of unmodified flows; performance/load testing

### Testing Types
| Type | Description |
|------|-------------|
| Functional | Does the feature work as specified? |
| Integration | Do systems communicate correctly end-to-end? |
| Failure Handling | What happens when things go wrong? |
| Data Validation | Are inputs validated at the boundary? |
| Exploratory | What else might break? |

### Defect Severity
| Severity | Definition | SLA |
|----------|------------|-----|
| P1 - Critical | Data loss, security breach, FCA-reportable | 24h |
| P2 - High | Major feature broken, workaround exists | 3 days |
| P3 - Medium | Partially broken, minor workaround | 1 week |
| P4 - Low | Cosmetic, no functional impact | Next sprint |

### Entry Criteria
- [ ] All P1/P2 dev defects closed
- [ ] UAT environment provisioned
- [ ] Test data prepared
- [ ] Build signed off by Diana

### Exit Criteria
- [ ] All P1/P2 defects closed
- [ ] Critical path scenarios passed
- [ ] Traceability matrix confirmed (100% coverage)
- [ ] Sign-off from Sarah Chen

## 3. Test Scenario Register

> **Activity:** Complete the register. For each Sprint 1 requirement, write: happy, alternative, failure path scenarios.
> Use **Prompt 4.1** to generate drafts from acceptance criteria.

In [ ]:
scenarios_data = {
    'ts_id': [
        'TS-FR001-001','TS-FR001-002','TS-FR001-003',
        'TS-FR002-001','TS-FR002-002','TS-FR002-003',
        'TS-FR003-001','TS-FR003-002','TS-FR003-003',
        'TS-FR004-001','TS-FR004-002','TS-FR004-003',
        'TS-FR005-001','TS-FR005-002','TS-FR005-003'
    ],
    'req_id': ['FR-001']*3 + ['FR-002']*3 + ['FR-003']*3 + ['FR-004']*3 + ['FR-005']*3,
    'type': ['Happy Path','Alternative Path','Failure Path']*5,
    'title': [
        'BO Declaration - All Fields Completed Successfully',
        'BO Declaration - Two 25% Owners (valid threshold)',
        'BO Declaration - Missing Ownership % Field (blocked)',
        'PEP Screening - No Match Found',
        'PEP Screening - Dow Jones API Timeout (fallback triggered)',
        'PEP Screening - True Positive Match (account paused)',
        'ID Verification - Experian Returns Pass',
        'ID Verification - Experian Returns Refer (manual review)',
        'ID Verification - Experian API Error (fallback activated)',
        'Audit Log - Entry Created on BO Declaration Submit',
        'Audit Log - Salesforce FSC Sync Failure (retry + alert)',
        'Audit Log - Record Integrity Hash Verification',
        'Risk Rating - UK Client, No PEP, ID Pass = Medium Risk',
        'Risk Rating - PEP Match Escalates to High Risk',
        'Risk Rating - Missing Data Defaults to Medium Pending'
    ],
    'cross_system': ['No']*3 + ['Yes - Dow Jones']*3 + ['Yes - Experian']*3 + ['Yes - Salesforce FSC']*3 + ['No']*3
}
df_scenarios = pd.DataFrame(scenarios_data)
print('Total scenarios:', len(df_scenarios))
print()
print(df_scenarios[['ts_id','req_id','type','cross_system']].to_string(index=False))

## 4. Cross-System Failure Scenarios

In [ ]:
cs_data = {
    'cs_id': ['CS-001','CS-002','CS-003','CS-004','CS-005'],
    'scenario': [
        'Experian API HTTP 503 During ID Verification',
        'Salesforce FSC Unreachable During BO Record Sync',
        'Dow Jones Returns PEP True Positive Match',
        'Audit DB Connection Timeout Mid-Transaction',
        'BO Declaration with SQL Injection Payload'
    ],
    'systems_affected': [
        'Web App -> Microservice -> Experian API',
        'Onboarding Microservice -> Salesforce FSC',
        'Microservice -> Dow Jones API -> Salesforce FSC',
        'Onboarding Microservice -> Audit DB',
        'Web App -> Onboarding Microservice'
    ],
    'expected_behaviour': [
        'System retries once. Second failure triggers Operations alert. Manual ID check fallback activated within 24h.',
        'Microservice retries 3x with exponential backoff. After 3 failures, record held locally. Operations alerted. No data loss.',
        'Account immediately flagged High Risk. Moved to Compliance Review queue. Email to Sarah Chen within 5 min. Onboarding paused.',
        'Write-ahead log preserves audit entry locally. Retry connection. If DB unavailable after 3 retries, alert Operations. No entry lost.',
        'Input validation rejects submission. Error shown to client. No data written. Security alert logged. Injection not possible.'
    ],
    'test_data': [
        'Simulated HTTP 503 via API mocking tool',
        'Salesforce FSC unavailable in test environment',
        'Known PEP individual in Dow Jones sandbox',
        'Simulate DB connection pool exhaustion',
        'Payloads: "Robert\'); DROP TABLE clients;--" and "<script>alert(1)</script>"'
    ]
}
df_cs = pd.DataFrame(cs_data)
print(df_cs[['cs_id','scenario','expected_behaviour']].to_string(index=False))

## 5. Traceability Matrix

In [ ]:
# REQUIREMENTS -> TEST SCENARIOS COVERAGE
print('TRACEABILITY MATRIX')
print('='*60)
for req_id in df_scenarios['req_id'].unique():
    subset = df_scenarios[df_scenarios['req_id']==req_id]
    happy = len(subset[subset['type']=='Happy Path'])
    alt = len(subset[subset['type']=='Alternative Path'])
    fail = len(subset[subset['type']=='Failure Path'])
    cross = len(subset[subset['cross_system']=='Yes'])
    print(f'{req_id}: {len(subset)} scenarios (Happy={happy}, Alt={alt}, Fail={fail}, CrossSystem={cross})')

all_reqs = set(df_fr['req_id'])
tested = set(df_scenarios['req_id'])
missing = all_reqs - tested
if missing:
    print()
    print('WARNING - No coverage:', missing)
else:
    print()
    print('All requirements have test scenario coverage.')

## 6. Export

In [ ]:
df_scenarios.to_csv('../outputs/test_scenario_register.csv', index=False)
df_cs.to_csv('../outputs/cross_system_scenarios.csv', index=False)
print('Exported: test_scenario_register.csv, cross_system_scenarios.csv')

## AI Prompt Library - Task 4

### Prompt 4.1: Test Scenario from Acceptance Criteria
```
Generate happy, alternative, and failure path scenarios from this acceptance criterion:
[paste acceptance criterion]

For each: Given [precondition] when [action] then [observable result].
Also: what would FAIL this test, what test data is needed.
```

### Prompt 4.2: Cross-System Failure Scenarios
```
Systems: Web App (React), Onboarding Microservice (Node.js/PostgreSQL),
Salesforce FSC, Experian Verify API, Dow Jones Risk Intel API, Audit DB.

Failures to scenario-ize:
A. Experian API HTTP 503 during ID verification
B. Dow Jones returns PEP true positive
C. Salesforce FSC unreachable during BO sync
D. Audit DB connection timeout
E. BO declaration with SQL injection
```